# Full-res U-Net TRAINING (GPU)

**Recommended — from-cache mode** (fast): attach the private cache datasets
(`climx-cache*` from the CPU cache notebook) as inputs; they are plain-copied to
/tmp (~30-40 min) and then fit + evaluate run with NO slow reads and NO
re-preprocessing. The raw 171 GB input is not needed.

**Fallback — build-in-session**: attach `climx-full-resolution-data` instead and run
cell 5; the cache is built in-session (~2-5 h on the slow input mount).

**Setup:** right panel → **Data → Add Input → Datasets** → add the cache datasets
(or the raw one for the fallback) → **Accelerator → GPU (T4 x2)**.

Ends with `FULLRES_TRAIN_OK` and the submission parquet path.

## 1. Clone the repo + install deps (~2 min, skips on reruns)

In [ ]:
import os, subprocess, sys, pathlib
repo = "/kaggle/working/climx"
if os.path.isdir(repo):
    r = subprocess.run(["git", "-C", repo, "pull", "--ff-only"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/nonewanit/climx-emulator.git", repo], check=True)
os.chdir(repo)
sys.path.insert(0, str(pathlib.Path(repo) / "src"))
subprocess.run(["find", repo, "-name", "__pycache__", "-type", "d",
                "-exec", "rm", "-rf", "{}", "+"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
r = subprocess.run(["git", "-C", repo, "log", "-1", "--oneline"], capture_output=True, text=True)
print("HEAD:", r.stdout.strip(), "— MUST be the commit None told you")
print("setup done")

## 2. Train from the cache (recommended)

In [ ]:
import os, pathlib
dirs = [str(p) for p in pathlib.Path("/kaggle/input").glob("*climx-cache*")]
print("cache datasets:", dirs)
assert dirs, "no climx-cache* datasets attached — use the fallback cell instead"
os.environ["CLIMX_CACHE_INPUT_DIRS"] = ":".join(dirs)
%run scripts/fullres_train.py --epochs 2 --batch 16

## 3. Fallback: build the cache in-session (slow input mount)

In [ ]:
%run scripts/fullres_train.py --subsample 3 --epochs 2 --batch 16

## 4. Submit the parquet (2/day team quota — coordinate!)

```bash
!kaggle competitions submit -c climx-main-track -f results/submissions/submission_unet_full.parquet -m "U-Net full-res"
```
Or download it (right panel → Output) and upload on the competition page.